# Recommendation Engine Notebook


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.configuration import AppConfig
from src.pipelines.portfolio_showcase_pipeline import run_portfolio_showcase_pipeline

sns.set_theme(style='whitegrid')
assets_dir = project_root / 'assets'
assets_dir.mkdir(parents=True, exist_ok=True)

config = AppConfig.from_env()
interactions = pd.read_csv(project_root / 'data' / 'raw' / 'interactions.csv')
items = pd.read_csv(project_root / 'data' / 'raw' / 'items.csv')
interactions.head()

In [ ]:
print('Interactions shape:', interactions.shape)
print('Items shape:', items.shape)
display(interactions.describe(include='all').T)
display(items.describe(include='all').T)

user_activity = interactions.groupby('user_id').size().rename('interaction_count')
item_popularity = interactions.groupby('item_id').size().rename('interaction_count')
display(user_activity.head())
display(item_popularity.sort_values(ascending=False).head(10))

In [ ]:
# EDA distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(user_activity, bins=25, kde=True, color='#2D728F', ax=axes[0])
axes[0].set_title('Interactions Per User')
axes[0].set_xlabel('Interaction Count')

top_items = item_popularity.sort_values(ascending=False).head(12)
labels = top_items.index.astype(str)
sns.barplot(x=labels, y=top_items.values, hue=labels, legend=False, palette='viridis', ax=axes[1])
axes[1].set_title('Top Items by Interactions')
axes[1].set_xlabel('Item')
axes[1].set_ylabel('Interactions')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(assets_dir / 'eda_distribution.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Missing values plot
missing_inter = interactions.isna().mean() * 100
missing_items = items.isna().mean() * 100
missing = pd.concat([missing_inter, missing_items]).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=missing.index.astype(str), y=missing.values, color='#4C78A8', ax=ax)
ax.set_title('Missing Values by Feature (%)')
ax.set_ylabel('Missing %')
ax.tick_params(axis='x', rotation=50)
fig.tight_layout()
fig.savefig(assets_dir / 'missing_values.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Correlation heatmap
corr_df = interactions.copy()
corr_df['interaction_type_code'] = corr_df['interaction_type'].astype('category').cat.codes
corr_cols = [c for c in ['user_id', 'rating', 'interaction_type_code'] if c in corr_df.columns]
corr = corr_df[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Interaction Feature Correlation')
fig.tight_layout()
fig.savefig(assets_dir / 'correlation_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# Generate all portfolio assets and real outputs
manifest = run_portfolio_showcase_pipeline(config)
manifest

In [ ]:
with (project_root / 'artifacts' / 'metrics.json').open('r', encoding='utf-8') as f:
    metrics = json.load(f)
with (project_root / 'artifacts' / 'output_samples.json').open('r', encoding='utf-8') as f:
    outputs = json.load(f)

display(pd.Series(metrics.get('metrics', {}), name='value').to_frame())
outputs

## Insights

- Hybrid recommendation balances collaborative relevance, semantic content similarity, and popularity fallback.
- Current data is sparse, so cold-start and popularity priors are important for stable ranking quality.
- All visual artifacts and real sample outputs are automatically generated for portfolio demonstration.

# Recommendation Engine EDA

Explore interaction density, item popularity, and sparsity patterns that drive recommender design.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
base_path = Path('../data/raw')
interactions = pd.read_csv(base_path / 'interactions.csv')
items = pd.read_csv(base_path / 'items.csv')
interactions.head()

In [ ]:
# Dataset overview
print(f'Interactions: {interactions.shape[0]} rows, {interactions.shape[1]} columns')
print(f'Items: {items.shape[0]} rows, {items.shape[1]} columns')
display(interactions.info())
display(items.info())
display(interactions.describe(include='all').T)

In [ ]:
# User activity distribution
activity = interactions.groupby('user_id').size().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.histplot(activity, bins=10, kde=True, color='steelblue')
plt.title('User activity distribution')
plt.xlabel('Interactions per user')
plt.tight_layout()
plt.show()

In [ ]:
# Popular items
popular = interactions.groupby('item_id').agg(interaction_count=('item_id', 'size'), avg_rating=('rating', 'mean')).sort_values(['interaction_count', 'avg_rating'], ascending=False).head(10)
popular = popular.join(items.set_index('item_id')[['title', 'genres']], how='left')
display(popular)
plt.figure(figsize=(10, 5))
sns.barplot(x=popular['interaction_count'], y=popular['title'], palette='viridis')
plt.title('Top popular items')
plt.xlabel('Interaction count')
plt.ylabel('Item')
plt.tight_layout()
plt.show()

In [ ]:
# Sparsity analysis
matrix = interactions.pivot_table(index='user_id', columns='item_id', values='rating', aggfunc='mean', fill_value=0)
sparsity = 1 - (matrix.astype(bool).sum().sum() / (matrix.shape[0] * matrix.shape[1]))
print(f'User-item matrix shape: {matrix.shape}')
print(f'Sparsity: {sparsity:.2%}')
plt.figure(figsize=(8, 5))
sns.heatmap(matrix.corr(), cmap='Blues', linewidths=0.3)
plt.title('Item correlation heatmap')
plt.tight_layout()
plt.show()

## Key Insights

- Sparse user-item matrices make fallback strategies necessary.
- Popular items should remain part of the hybrid scorer.
- Collaborative and content-based signals are both useful when interactions are limited.

In [ ]:
# Recommendation quality diagnostics: catalog coverage and long-tail concentration
item_popularity = interactions.groupby('item_id').size().sort_values(ascending=False)
coverage_ratio = item_popularity.size / items['item_id'].nunique()
head_share = item_popularity.head(max(1, int(0.2 * len(item_popularity)))).sum() / item_popularity.sum()

print(f'Catalog coverage in interactions: {coverage_ratio:.2%}')
print(f'Top-20% items share of interactions: {head_share:.2%}')

plt.figure(figsize=(10, 4))
item_popularity.reset_index(drop=True).plot(kind='line', color='darkorange')
plt.title('Item popularity curve (head vs long-tail)')
plt.xlabel('Items sorted by popularity rank')
plt.ylabel('Interaction count')
plt.tight_layout()
plt.show()

## Recommendation Quality Insights

- A high head-share indicates popularity bias risk, so hybrid blending should preserve some exploration via content and collaborative signals.
- Lower catalog coverage usually means sparse feedback; cold-start defaults and popularity priors become critical.
- These EDA checks justify tuning hybrid weights rather than hardcoding them.